<a href="https://colab.research.google.com/github/Areej973/Research-Paper-Assistant-using-RAG/blob/main/Research_Paper_Assistant_using_RAG1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Research Paper Assistant using RAG

In [ ]:
!pip install faiss-cpu sentence-transformers transformers datasets==2.14.5

**Import Libraries**

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd

**Load and Explore Research Data**

**Scenario**

You are building an assistant for AIResearchHub, a company that wants to summarize or answer questions about new computer-science papers.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("armanc/scientific_papers","arxiv",split="train[:100]")

import pandas as pd
df = pd.DataFrame(dataset)[["abstract"]]
df.head()

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

,abstract
0,additive models play an important role in sem...
1,"we have studied the leptonic decay @xmath0 , ..."
2,"in 84 , 258 ( 2000 ) , mateos conjectured tha..."
3,the effect of a random phase diffuser on fluc...
4,with a special intention of clarifying the un...


In [ ]:
print("Number of papers:",len(df))
print("\nSample abstract:\n")
print(df['abstract'][0][:500],"...")

Number of papers: 100

Sample abstract:

 additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel , provided the assumption of an additive model is valid . 
 additionally , a concrete example is pr ...


**Create Document Embeddings**

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
documents = df['abstract'].tolist()
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
print("Embeddings shape:",doc_embeddings.shape)

Embeddings shape: (100, 384)


**Build a FAISS Index for Retrieval**

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)
print(f"FAISS index contains {index.ntotal} abstracts.")


FAISS index contains 100 abstracts.


**Retrieval Function**

In [ ]:
def retrieve(query, top_k=3):
   query_emb = embedding_model.encode([query], normalize_embeddings=True)
   distances, indices = index.search(query_emb, top_k)
   retrieved_docs = [documents[i] for i in indices[0]]
   return retrieved_docs

**Test**

In [ ]:
query = "What are recent advancements in reinforcement learning?"
results = retrieve(query)

for i, abs_text in enumerate(results, start=1):
  print(f"\n Abstract {i}:\n {abs_text[:400]}...")


 Abstract 1:
  synaptic memory is considered to be the main element responsible for learning and cognition in humans . 
 although traditionally non - volatile long - term plasticity changes have been implemented in nanoelectronic synapses for neuromorphic applications , recent studies in neuroscience have revealed that biological synapses undergo meta - stable volatile strengthening followed by a long - term st...

 Abstract 2:
  additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favourably in particular in high dimensions to recent results on optimal learning rates for purely nonparametric regularized kernel based quantile regression using the gaussian radial basis function kernel...

 Abstract 3:
  we propose a tuner , suitable for adaptive control and ( in its discrete - time version ) adaptive filtering applications , that sets the second d

**Add a Generation Model (FLAN-T5)**

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
